# 🎨 face2sketch — Phase 1 v3: Photo → Sketch (Kaggle)

**Retrain pix2pix with D stabilization: spectral norm + noise injection.**

### Why v3
- v1/v2: D died at epoch ~30 → outputs were blurry photocopies
- v3: spectral norm + noise injection → D stays alive → pencil-drawing texture

### Settings
- 200 epochs, batch=16, 256×256, λ_L1=100
- D: spectral norm + noise σ=0.05
- G_lr=2e-4, D_lr=1e-4

In [ ]:
# @title 1. Clone Repo + Install
import os, sys, shutil
BASE = '/kaggle/working/face2sketch'
os.makedirs(BASE, exist_ok=True)
os.chdir('/kaggle/working')

!git clone https://github.com/weseegod/face2sketch.git {BASE} 2>/dev/null || true
os.chdir(BASE)
!git pull origin main

!pip install -q tqdm matplotlib pillow pyyaml einops
!mkdir -p checkpoints samples outputs

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print('\n✅  Setup complete.')

os.environ['FACE2SKETCH_BASE'] = BASE

In [ ]:
# @title 2. Find + Extract Data
import os, shutil

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

# Find data.zip anywhere under /kaggle/input/
data_zip = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'data.zip' in files:
        data_zip = os.path.join(root, 'data.zip')
        break

if data_zip:
    print(f'📦  Unzipping {data_zip} ...')
    !unzip -o {data_zip} -d {BASE}/ 2>&1 | tail -3
else:
    print('❌  data.zip not found in /kaggle/input/')

# Verify
ds = len(os.listdir(f'{BASE}/data/dataset/photos')) if os.path.isdir(f'{BASE}/data/dataset/photos') else 0
ts = len(os.listdir(f'{BASE}/data/test/photos')) if os.path.isdir(f'{BASE}/data/test/photos') else 0
print(f'✅  Training: {ds//2} pairs  |  Test: {ts//2} pairs')

if ds == 0:
    print('⛔  No data found. Fix dataset and re-run.')
    raise SystemExit(1)

In [ ]:
# @title 3. Train — 200 epochs (~1.5 hr)
import os
assert torch.cuda.is_available(), "⚠️  Enable GPU in Notebook settings"

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

print("🎯  Training Phase 1 v3 (spectral norm + noise injection)")
print("    λ_L1=100 | D spectral norm | noise σ=0.05 | G_lr=2e-4")
print(f"    322 pairs  |  200 epochs  |  256×256\n")

!python src/train.py --mode train \
    --device cuda \
    --name pix2pix_v3_

print("\n✅  Training complete → checkpoints/pix2pix_v3_best.pt")

In [ ]:
# @title 4. Evaluate
import os, glob

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

ckpt = 'checkpoints/pix2pix_v3_best.pt'
if not os.path.exists(ckpt):
    candidates = sorted(glob.glob('checkpoints/pix2pix_v3_*.pt'), reverse=True)
    for c in candidates:
        if os.path.getsize(c) > 100000:
            ckpt = c; break

if not os.path.exists(ckpt):
    print('❌  No checkpoint found.')
elif not os.path.exists('data/test/photos'):
    print('⚠️  No test data.')
else:
    !python src/evaluate.py --checkpoint {ckpt} --mode quick --device cuda
    !python src/evaluate.py --checkpoint {ckpt} --mode full --device cuda
    print('\n📸  Check outputs/ folder for visual results.')
    print('    phase1_quick_eval.png — 10 photos + generated')
    print('    phase1_full_eval.png  — 8 generated vs ground-truth')

In [ ]:
# @title 5. Save to Kaggle Output
import os, shutil

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
OUT  = '/kaggle/working'

for name in ['pix2pix_v3_best.pt', 'pix2pix_v3_final.pt']:
    src = os.path.join(BASE, 'checkpoints', name)
    if os.path.exists(src):
        os.makedirs(f'{OUT}/checkpoints', exist_ok=True)
        shutil.copy(src, f'{OUT}/checkpoints/{name}')
        size = os.path.getsize(src) / (1024*1024)
        print(f'✅  {name}  ({size:.0f}MB)')

for item in ['samples', 'outputs']:
    src = os.path.join(BASE, item)
    dst = os.path.join(OUT, item)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'✅  {item}/')

print('\n📥  Download from Output tab: best.pt + final.pt + samples/ + outputs/')
print('🎉  Done!')
